# Phase 6 — Suivi des portefeuilles

## Pourquoi cette phase est importante

Les trois portefeuilles sont déjà construits. Ils partagent le même univers et les mêmes rendements de titres. Cette phase regarde où va le poids, et à quel point il est concentré. Elle ne recalcule pas les poids.

Le suivi répond aux questions du mandat.

- Quelle part du portefeuille est dans chaque secteur, chaque titre, chaque zone et chaque devise de cotation.
- Quelle part représentent les dix premières lignes, et la concentration qui en résulte.
- Les plafonds de l'IPS ont-ils tenu à chaque mois de la période : émetteur, secteur, zone, devise, et la bande de lignes des deux livres optimisés.
- La rotation brute d'un trimestre est-elle restée au plus à 12,5 %, et la somme de quatre trimestres au plus à 50 %.

## Les livrables

| Livrable | Fichier | Rôle |
|---|---|---|
| Documentation | `documentation/06_portfolio_monitoring.ipynb` | La méthode, les formules, les fichiers produits |
| Rapport mensuel | `reports/monitoring/06_rapport_mensuel.ipynb` | La situation au dernier mois, et le contrôle des plafonds sur toute la période |
| Dashboard | `notebooks/06_dashboard_monitoring.ipynb` et `reports/monitoring/dashboard.html` | Les graphiques des trois portefeuilles |

La vérification automatique est à part, dans `tests/monitoring/phase_06_validation.ipynb`. Elle ne remplace aucun livrable.

## Situation du mois et suivi de la période

Le rapport mensuel porte la date du dernier mois, lue dans `portfolio_metrics.csv`. Il décrit le portefeuille tenu ce mois-là : secteur, titre, zone, devise, dix premières lignes et concentration. Ce sont les expositions écrites dans la politique.

Chaque mois de la période a son propre poids. Il est fixé avec les 60 mois déjà connus, puis tenu pendant le trimestre. `concentration_history.csv` mesure la concentration à chaque mois. `limit_watch.csv` compare, à chaque mois, le plus grand émetteur, le plus grand secteur, la plus grande zone et la plus grande devise au plafond de l'IPS. `turnover.csv` mesure la rotation brute au début de chaque trimestre.

Le rendement d'un mois vient du poids de ce mois. La performance cumulée enchaîne ces décisions. Elle ne reprend pas le poids du dernier mois sur toute la période.


## Sources du registre

Le suivi part des portefeuilles déjà publiés. Il ne réestime pas les poids. Pour la situation, il relit la feuille de positions du benchmark. Pour l'historique, il relit les mêmes cours et le même SARON que la phase 5, afin de retrouver le rendement de chaque titre.

| Fichier lu | Origine | Rôle dans cette phase |
|---|---|---|
| `data/04_portfolios/portfolio_weights.csv` | sortie de la phase 5 | Les poids des trois stratégies |
| `data/04_portfolios/portfolio_metrics.csv` | sortie de la phase 5 | La période, le nombre de titres, le nombre effectif déjà publié |
| `data/04_portfolios/portfolio_monthly_returns.csv` | sortie de la phase 5 | Le rendement mensuel officiel. La somme des contributions doit le retrouver |
| `data/02_processed/market/market_prices_chf.csv` | `DS-003` et `DS-004` | Le rendement de chaque titre, pour la contribution des secteurs |
| `data/03_universe/eligible/chf_portfolio_universe.csv` | `DS-002` et `DS-004` | La même liste que la phase 5, pour reconstruire le panneau de rendements |
| `data/02_processed/macro/saron.csv` | `DS-005` | La même période que la phase 5 |
| `data/01_raw/benchmark/benchmark_holdings.xls` | `DS-000` | `Sector`, `Asset Class`, `Location` |

Le module ne contient pas l'identifiant `DS-000`. Il retrouve la feuille par le jeu qui produit `security_candidates.csv`, c'est-à-dire `DS-001`, dont l'entrée est le benchmark. Si un ticker y est répété, le calcul s'arrête.

`DS-006` n'est pas lu. L'ODD ne classe ni un secteur, ni une zone, ni une exposition.


## Entrées

Le module lit les poids publiés, la classification du benchmark, et les fichiers qui reconstruisent le rendement de chaque titre. Il ne réécrit ni les cours ni les poids.

| Rôle | Fichier | Colonnes utilisées |
|---|---|---|
| Poids | `data/04_portfolios/portfolio_weights.csv` | `strategy`, `month`, `yahoo_symbol`, `ticker`, `name`, `weight` |
| Période et contrôle | `data/04_portfolios/portfolio_metrics.csv` | `strategy`, `names`, `window_start`, `window_end`, `maximum_weight`, `effective_holdings` |
| Rendement officiel | `data/04_portfolios/portfolio_monthly_returns.csv` | `month`, `strategy`, `portfolio_return` |
| Cours de l'historique | `data/02_processed/market/market_prices_chf.csv` | `symbol`, `timestamp`, `close_chf` |
| Univers de l'historique | `data/03_universe/eligible/chf_portfolio_universe.csv` | `yahoo_symbol`, `chf_status` |
| Période SARON | `data/02_processed/macro/saron.csv` | `series_id`, `date`, `value`, `rate_currency` |
| Classification | `data/01_raw/benchmark/benchmark_holdings.xls`, jeu `DS-000` | `Ticker`, `Sector`, `Asset Class`, `Location` |

Aucun identifiant de jeu n'est écrit dans le module. Le fichier de positions est celui que le registre désigne déjà comme entrée des candidats. Si `Ticker` n'est pas unique dans cette feuille, le calcul s'arrête : un même ticker ne peut pas porter deux secteurs.

Un titre du portefeuille absent de cette feuille reste dans le portefeuille. Son secteur, son pays et sa classe d'actif deviennent `Non classé`. Le poids n'est pas retiré, et les totaux continuent de faire 1.

## Secteur, zone, exposition

Le secteur est le texte de la colonne `Sector`, sans traduction. Une valeur vide devient `Non classé`.

Le pays est le texte de `Location`. Une valeur vide, ou `--`, devient `Non classé`. La zone est un regroupement de ce pays :

| Zone | Pays publiés par la feuille de positions |
|---|---|
| Asie-Pacifique | Australia, China, Hong Kong, India, Japan, Korea (South), Malaysia, Singapore, Taiwan, Thailand |
| Europe | Austria, Belgium, Denmark, European Union, France, Germany, Italy, Norway, Portugal, Spain, Sweden, Switzerland, United Kingdom |
| Amériques | Brazil, Canada, Chile, Mexico, United States |
| Moyen-Orient et Afrique | Israel, Saudi Arabia, South Africa |

Un pays qui n'est pas dans cette table reste dans le portefeuille, dans la zone `Non classé`. Le journal technique le signale. Le calcul ne s'arrête pas, et le pays n'est pas rangé dans une zone au jugé.

L'exposition en devise est la devise de cotation, dans `exposure.csv`. Elle sert le plafond de devise. La politique autorise les actions cotées et les ETF actions cotés. Les livres publiés viennent de la feuille de positions du proxy, qui ne contient que des actions. Le suivi ne fabrique pas une coupe actions, obligations et ETF : ces trois cases ne sont pas les expositions du mandat.

Une ligne dont le poids est inférieur ou égal à `1e-8` est de la poussière numérique de l'optimiseur, pas une position. Elle reste dans le fichier ligne à ligne et dans l'indice de Herfindahl. Elle sort des tableaux de secteur, de zone et de devise. Le seuil est très au-dessus de cette poussière, qui est de l'ordre de `1e-17`, et très au-dessous d'une ligne réellement publiée.


## Formules

Soit $w_i$ le poids publié du titre $i$, et $N$ le nombre de titres de l'univers commun. Ce nombre comprend les lignes que l'optimiseur a mises à zéro. Les trois portefeuilles ont le même $N$.

L'allocation d'un groupe $G$ (un secteur, une zone ou une devise) est

$$
w(G) = \sum_{i \in G} w_i
$$

Les groupes d'une même coupe se partagent le portefeuille. Leur somme vaut 1, à la poussière numérique près. Le module refuse de renormaliser un fichier de poids qui ne somme pas déjà à 1.

Les dix premières lignes sont les plus grands poids. À poids égal, l'ordre suit `yahoo_symbol`. S'il y a moins de dix lignes au-dessus du seuil, le top 10 est la somme de ces lignes.

$$
w_{(10)} = \sum_{k=1}^{\min(10, N_+)} w_{(k)}
$$

$N_+$ est le nombre de lignes au-dessus du seuil. Pour l'équipondéré, tous les poids sont égaux, donc $w_{(10)} = 10 / N$. Les dix noms affichés sont alors les dix premiers symboles, pas dix titres plus gros que les autres.

L'indice de Herfindahl utilise tous les poids publiés, y compris les zéros :

$$
H = \sum_{i=1}^{N} w_i^2
$$

Le nombre effectif de titres est

$$
N_{eff} = \frac{1}{H}
$$

Il vaut $N$ quand tous les poids sont égaux. Il baisse quand le poids se concentre. La phase 5 publie déjà $N_{eff}$. Cette phase le recalcule et s'arrête si les deux valeurs divergent.

Le multiple de concentration compare le portefeuille à l'équipondéré de même univers :

$$
C = N \, H
$$

$C = 1$ quand chaque titre pèse $1/N$. $C$ augmente quand le poids se concentre. Un multiple de 4 veut dire que le portefeuille est aussi concentré qu'un équipondéré de $N/4$ titres.

Ces mesures de concentration sont recalculées à chaque mois, sur le poids publié de ce mois. Le fichier de concentration du rapport est la ligne du dernier mois. Elle doit retrouver le nombre effectif déjà publié par la phase 5.

Le suivi des plafonds retient, chaque mois, la ligne la plus lourde et le groupe le plus lourd. Un indicateur vaut 1 quand ce poids dépasse le plafond de l'IPS, et 0 quand il reste dedans. Le plafond d'émetteur est celui d'une ligne de cotation.

La bande de 25 à 40 lignes vaut pour les deux portefeuilles optimisés. L'équipondéré tient toute la liste, donc la bande ne le marque pas. Un mois optimisé en dehors de la bande est marqué dans `limit_watch.csv`.

La rotation brute d'un trimestre est la somme des valeurs absolues des changements de poids par rapport au livre du trimestre précédent. Le suivi la calcule en janvier, avril, juillet et octobre, les mois où la construction fixe un nouveau livre. Le premier livre n'a pas de prédécesseur, donc le premier mois de la période n'a pas de rotation. Un mois gardé à l'intérieur du trimestre a les mêmes poids. Le plafond trimestriel est 12,5 %. La somme des quatre derniers trimestres se compare à 50 %. Un livre posé sur le plafond reste dedans : le dépassement commence au-dessus.


## Historique de performance

La situation décrit les poids du dernier mois. L'historique décrit ce que le poids de chaque mois a produit pendant la période.

Le poids d'un trimestre est fixé avec les soixante mois déjà connus. Il est tenu trois mois. La construction le publie sous les plafonds de l'IPS : 5 % par émetteur, 40 % par secteur, 50 % par zone, 40 % par devise. Le suivi reprend chaque mois et marque un dépassement dans `limit_watch.csv`. La situation publiée est celle du mois du rapport.

Le rendement du titre $i$ au mois $t$ est $r_{i,t}$. Avec le poids cible $w_i$, sa contribution est

$$
c_{i,t} = w_i \, r_{i,t}
$$

La contribution d'un secteur est la somme des contributions de ses titres. Au mois $t$, la somme des secteurs est le rendement du portefeuille. La série cumulée additionne ces contributions. Elle n'est pas la valeur composée du portefeuille.

Un placement de 1 CHF part de 1 :

$$
W_t = \prod_{u \le t} (1 + R_u)
$$

Le drawdown est

$$
D_t = \frac{W_t}{\max_{u \le t} W_u} - 1
$$

$R_u$ est le rendement du portefeuille au mois $u$, avec le poids fixé à la fin du mois précédent. La performance cumulée enchaîne ces décisions. Elle n'est pas le résultat d'un poids unique gardé dix ans.

## Fichiers produits

Tous sont sous `data/05_monitoring/`. Chaque exécution les remplace.

| Fichier | Contenu |
|---|---|
| `allocation_by_holding.csv` | Une ligne par stratégie et par titre, avec secteur, pays, zone et exposition |
| `allocation_by_sector.csv` | Poids et nombre de lignes par secteur |
| `allocation_by_zone.csv` | Poids et nombre de lignes par zone, au mois du rapport |
| `exposure.csv` | Devise de cotation, au mois du rapport |
| `concentration.csv` | Poids maximum, top 10, Herfindahl, nombre effectif, multiple, au mois du rapport |
| `concentration_history.csv` | Les mêmes mesures, à chaque mois |
| `limit_watch.csv` | À chaque mois, le plus grand émetteur, secteur, zone et devise, et le dépassement de plafond |
| `turnover.csv` | Au début de chaque trimestre, la rotation brute et la somme des quatre derniers trimestres |
| `top10_positions.csv` | Les dix premières lignes, au mois du rapport |
| `monthly_report.csv` | Une ligne par stratégie, datée du dernier mois de la période |
| `portfolio_path.csv` | Chaque mois : rendement, performance d'un placement de 1 CHF, drawdown |
| `sector_contribution.csv` | Contribution du secteur au rendement du mois, puis cumul |

La commande qui reconstruit ces fichiers seuls est

```
python -m src.portfolio_monitoring.build_monitoring
```

`python -m src.pipeline.refresh` le fait après la construction des portefeuilles, sur les poids qui viennent d'être estimés. La commande écrit aussi `reports/monitoring/dashboard.html`.

La cellule suivante lit le rapport déjà publié et affiche le mois, la période et les dépassements de plafond. Elle ne fixe aucune de ces valeurs.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.config import MONITORING_DATA

report = pd.read_csv(MONITORING_DATA / "monthly_report.csv")
row = report.iloc[0]
print(f"Mois du rapport : {row.report_month}")
print(f"Période : {row.window_start} à {row.window_end}")
print(f"Stratégies : {report['strategy'].nunique()}")
for item in report.itertuples(index=False):
    print(
        f"{item.strategy} : {int(item.invested_names)} lignes investies, "
        f"top 10 {item.top_10_weight:.1%}, "
        f"devise principale {item.largest_currency} {item.largest_currency_weight:.1%}."
    )

limits = pd.read_csv(MONITORING_DATA / "limit_watch.csv")
flags = ["issuer_over_cap", "sector_over_cap", "zone_over_cap", "currency_over_cap"]
print("Mois au-dessus d'un plafond")
for strategy in report["strategy"]:
    book = limits.loc[limits["strategy"].eq(strategy)]
    print(f"{strategy} : {int(book[flags].to_numpy().sum())} sur {len(book)} mois")
print("Mois hors de la bande de 25 à 40 lignes")
for strategy in report["strategy"]:
    book = limits.loc[limits["strategy"].eq(strategy)]
    if strategy == "equal_weight":
        print(f"{strategy} : la bande ne s'applique pas")
    else:
        print(
            f"{strategy} : {int(book['positions_outside_band'].sum())} sur {len(book)} mois"
        )

turnover = pd.read_csv(MONITORING_DATA / "turnover.csv")
print("Rotation brute trimestrielle")
for strategy in report["strategy"]:
    book = turnover.loc[turnover["strategy"].eq(strategy)]
    print(
        f"{strategy} : maximum {book['gross_turnover'].max():.2%}, "
        f"{int(book['over_quarterly_cap'].sum())} trimestre au-dessus de 12,5 %, "
        f"quatre trimestres au plus {book['trailing_annual_turnover'].max():.2%}, "
        f"{int(book['over_annual_cap'].sum())} fenêtre au-dessus de 50 %."
    )


Mois du rapport : 2026-10
Période : 2016-11 à 2026-10
Stratégies : 3
equal_weight : 29 lignes investies, top 10 34.5%, devise principale JPY 17.2%.
minimum_volatility : 26 lignes investies, top 10 50.0%, devise principale JPY 23.5%.
maximum_sharpe : 25 lignes investies, top 10 49.0%, devise principale USD 21.3%.
Mois au-dessus d'un plafond
equal_weight : 0 sur 120 mois
minimum_volatility : 0 sur 120 mois
maximum_sharpe : 0 sur 120 mois
Mois hors de la bande de 25 à 40 lignes
equal_weight : la bande ne s'applique pas
minimum_volatility : 0 sur 120 mois
maximum_sharpe : 0 sur 120 mois
Rotation brute trimestrielle
equal_weight : maximum 0.00%, 0 trimestre au-dessus de 12,5 %, quatre trimestres au plus 0.00%, 0 fenêtre au-dessus de 50 %.
minimum_volatility : maximum 12.50%, 0 trimestre au-dessus de 12,5 %, quatre trimestres au plus 50.00%, 0 fenêtre au-dessus de 50 %.
maximum_sharpe : maximum 12.50%, 0 trimestre au-dessus de 12,5 %, quatre trimestres au plus 50.00%, 0 fenêtre au-dessus de 

# Dictionnaire des termes de cette note

Ce dictionnaire reprend les termes employés dans cette note. Il n'en ajoute pas. Les formules restent dans la section qui les écrit.

**Allocation.** Somme des poids d'un groupe : un secteur, une zone ou une devise.

**Benchmark.** Feuille de positions du proxy investissable, `DS-000`. Elle fournit le secteur, le pays et la classe d'actif. La référence stratégique est l'indice : elle n'est pas cette feuille.

**Contribution.** Poids d'un titre multiplié par son rendement du mois. La somme des contributions de secteur retrouve le rendement du portefeuille.

**Drawdown.** Baisse de la performance cumulée depuis le plus haut déjà atteint.

**Émetteur.** Société du titre. Le plafond de 5 % rappelé ici vient de la construction.

**Exposition.** Dans cette note, la devise de cotation. Le fichier est `exposure.csv`. Les autres expositions du mandat sont le secteur et la zone.

**Herfindahl.** Somme des carrés des poids, y compris les poids nuls.

**Multiple de concentration.** Herfindahl multiplié par le nombre de titres. Il vaut 1 quand tous les poids sont égaux.

**Nombre effectif de titres.** Inverse du Herfindahl. Il baisse quand le poids se concentre.

**Non classé.** Secteur, pays, zone ou classe d'actif absent de la feuille de positions. Le poids reste dans le portefeuille.

**Plafond.** Limite de l'IPS sur une ligne d'émetteur, un secteur, une zone ou une devise. `limit_watch.csv` marque le mois où le poids publié la dépasse.

**Rotation brute.** Somme des valeurs absolues des changements de poids entre le livre d'un trimestre et celui du trimestre précédent. `turnover.csv` la compare à 12,5 %, et la somme de quatre trimestres à 50 %.

**ODD.** Évaluation non lue dans cette phase. Elle ne classe ni un secteur, ni une zone, ni une exposition.

**Performance cumulée.** Valeur d'un placement de 1 CHF qui enchaîne les rendements mensuels.

**Poussière numérique.** Poids inférieur ou égal à 1e-8, résidu de l'optimiseur. Il reste dans le Herfindahl et sort des tableaux de secteur, de zone et d'exposition.

**SARON.** Fichier de taux relu pour retrouver la même période que la construction.

**Secteur.** Texte de la colonne `Sector`, sans traduction.

**Situation.** Portrait du portefeuille au dernier mois, pas l'allocation de chaque mois passé.

**Top 10.** Somme des dix plus grands poids au-dessus du seuil.

**Zone.** Regroupement du pays lu dans `Location` : Asie-Pacifique, Europe, Amériques, Moyen-Orient et Afrique.
